# Phase 1: Record Expert Activation Traces

**Runtime:** A100 GPU (Colab Pro/Pro+)

This notebook:
1. Installs dependencies (torch, transformers, accelerate)
2. Uploads/mounts the MoE-Sched package
3. Loads Mixtral-8×7B with CPU offloading
4. Hooks into MoE layers to capture router outputs
5. Runs inference on sample prompts
6. Records expert activation traces to Google Drive

**Important:** Select Runtime → Change runtime type → A100 GPU before running.

## 0. Environment Setup

**Before running any notebook**, prepare your Google Drive:

1. Create folder `My Drive/moe-sched-paper/`
2. Zip the `conference-paper/` directory from your local machine:
   ```
   cd homework4/conference-paper
   zip -r moe_sched.zip moe_sched/
   ```
3. Upload `moe_sched.zip` to `My Drive/moe-sched-paper/`

The notebooks will unzip it to `/content/` and add it to the Python path.
Traces and results are saved back to `My Drive/moe-sched-paper/` so they
persist across sessions.

**Notebook execution order:**
1. `01_trace_recording` — Record Mixtral expert traces
2. `02_profile_dispatch` — Profile dispatch overhead (Python + Cython)
3. `03_baselines` — vLLM / MoE-Infinity throughput
4. `04_full_evaluation` — All policies × all traces → tables & figures
5. `05_deepseek_traces` — Record DeepSeek-V2-Lite traces (separate session)
6. `06_e2e_throughput` — End-to-end Mixtral throughput with hooks

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate lark>=1.1 huggingface_hub

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert 'A100' in torch.cuda.get_device_name(0) or True, "Expected A100 — adjust if using different GPU"

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/moe-sched-paper'
os.makedirs(f'{DRIVE_ROOT}/traces', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# Unzip moe_sched from Google Drive
!unzip -qo /content/drive/MyDrive/moe-sched-paper/moe_sched.zip -d /content/

# Add to Python path
import sys
sys.path.insert(0, '/content')

# Verify import
from moe_sched import MoESched
print('MoE-Sched imported successfully')

## 1. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_NAME = 'mistralai/Mixtral-8x7B-Instruct-v0.1'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model: {MODEL_NAME}')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map='auto',  # auto CPU/GPU split
    # cache_dir=f'{DRIVE_ROOT}/checkpoints',  # cache on Drive to avoid re-download
)
print(f'Loaded in {time.time() - t0:.1f}s')
print(f'Device map: {model.hf_device_map}')

## 2. Baseline Inference (No Hooks)

In [ ]:
# Measure baseline throughput without any hooks
test_prompts = [
    'Explain the difference between transformers and RNNs.',
    'Write a Python function to sort a list using merge sort.',
    'What are the main challenges in training large language models?',
]

print('Baseline inference (no hooks):')
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    num_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    tok_per_sec = num_tokens / elapsed
    print(f'  {num_tokens} tokens in {elapsed:.2f}s = {tok_per_sec:.1f} tok/s')

## 3. Hook Into MoE Layers

In [ ]:
# Identify the MoE layer structure
# Mixtral uses: model.model.layers[i].mlp (MixtralSparseMoeBlock)
# Children: gate (MixtralTopKRouter), experts (MixtralExperts)
# num_experts inferred from gate.weight.shape[0]

layers = model.model.layers
print(f'Number of layers: {len(layers)}')

# Inspect first MoE layer
moe_block = layers[0].mlp
print(f'MoE block type: {type(moe_block).__name__}')
print(f'Gate type: {type(moe_block.gate).__name__}')
print(f'Experts type: {type(moe_block.experts).__name__}')
print(f'Top-k: {moe_block.top_k}')
print(f'Gate shape: {moe_block.gate.weight.shape}')  # (num_experts, hidden_size)

num_experts = moe_block.gate.weight.shape[0]
num_layers = len(layers)
print(f'\nModel: {num_layers} layers × {num_experts} experts (top-{moe_block.top_k})')

In [ ]:
import json
from collections import defaultdict

# Storage for captured expert selections
trace_data = []
token_counter = [0]  # mutable counter for closure

def make_gate_hook(layer_idx):
    """Hook on gate module to capture router selections."""
    def hook_fn(module, input, output):
        # Gate output: (router_logits, topk_weights, topk_ids)
        router_logits, topk_weights, topk_ids = output

        # Record each token's expert selections
        for i in range(topk_ids.shape[0]):
            trace_data.append({
                't': token_counter[0] + i,
                'l': layer_idx,
                'e': topk_ids[i].cpu().tolist(),
                's': [round(s, 4) for s in topk_weights[i].cpu().tolist()],
            })
    return hook_fn

# Install hooks on all gate modules
hooks = []
for idx, layer in enumerate(layers):
    gate = layer.mlp.gate
    h = gate.register_forward_hook(make_gate_hook(idx))
    hooks.append(h)

print(f'Installed hooks on {len(hooks)} gate modules')

## 4. Run Inference With Hooks

In [ ]:
# Run inference and capture traces
trace_data.clear()
token_counter[0] = 0

prompts = [
    'Explain the difference between transformers and RNNs.',
    'Write a Python function to sort a list using merge sort.',
    'What are the main challenges in training large language models?',
    'Describe the architecture of a Mixture-of-Experts model.',
    'How does expert caching improve MoE inference performance?',
]

print(f'Running inference on {len(prompts)} prompts...')
for i, prompt in enumerate(prompts):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    token_counter[0] = 0  # reset per prompt
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )
    num_tokens = outputs.shape[1]
    print(f'  Prompt {i+1}: {num_tokens} tokens, {len(trace_data)} trace entries so far')

print(f'\nTotal trace entries: {len(trace_data)}')

# Remove hooks to restore original behavior
for h in hooks:
    h.remove()
print('Hooks removed')

## 5. Analyze and Save Traces

In [ ]:
# Quick analysis
from collections import Counter

expert_counts = Counter()
for entry in trace_data:
    for e in entry['e']:
        expert_counts[e] += 1

print('Expert activation distribution (top 10):')
for expert_id, count in expert_counts.most_common(10):
    pct = 100 * count / sum(expert_counts.values())
    print(f'  Expert {expert_id}: {count} activations ({pct:.1f}%)')

print(f'\nTotal unique experts activated: {len(expert_counts)}')
print(f'Total activations: {sum(expert_counts.values())}')

In [ ]:
# Save trace to Google Drive
import json

trace_path = f'{DRIVE_ROOT}/traces/mixtral_sample.jsonl'

with open(trace_path, 'w') as f:
    # Header
    f.write(json.dumps({
        'model_name': MODEL_NAME,
        'num_layers': num_layers,
        'num_experts': num_experts,
        'top_k': 2,
        'num_entries': len(trace_data),
    }) + '\n')
    # Entries
    for entry in trace_data:
        f.write(json.dumps(entry) + '\n')

print(f'Trace saved to: {trace_path}')
print(f'File size: {os.path.getsize(trace_path) / 1e6:.1f} MB')

## 6. Test MoE-Sched Hooks on Real Model

In [ ]:
# Compile a DSL policy and test it against the recorded trace
from moe_sched import MoESched
from moe_sched.compiler import compile_policy
from moe_sched.runtime.hooks import build_hook

# Define a simple LRU policy via fluent builder
sched = MoESched()
ir = (
    sched.build('lru_test')
    .cache(capacity=4, eviction='lru')
    .prefetch(strategy='none')
    .schedule(mode='gpu_only')
    .done()
)

policy = compile_policy(ir)
hook = build_hook(policy)

# Replay the recorded trace through the hook
print(f'Replaying {len(trace_data)} trace entries through LRU policy...')
for entry in trace_data:
    hook.on_layer(
        layer_idx=entry['l'],
        selected_experts=entry['e'],
        scores=entry.get('s'),
    )

stats = hook.stats_snapshot()
print(f'\nResults on real Mixtral traces:')
print(f'  Cache hits:   {stats["cache"]["hits"]}')
print(f'  Cache misses: {stats["cache"]["misses"]}')
hit_rate = stats['cache']['hits'] / max(1, stats['cache']['hits'] + stats['cache']['misses'])
print(f'  Hit rate:     {hit_rate:.1%}')
print(f'  Evictions:    {stats["cache"]["evictions"]}')

In [ ]:
# Compare multiple policies on the same real trace
from moe_sched.benchmark.policies import get_dsl_policies

policies = get_dsl_policies()

print(f'{"Policy":<25} {"Hit Rate":>10} {"Evictions":>10}')
print('-' * 47)

for name, compiled in policies.items():
    hook = build_hook(compiled)
    for entry in trace_data:
        hook.on_layer(
            layer_idx=entry['l'],
            selected_experts=entry['e'],
            scores=entry.get('s'),
        )
    stats = hook.stats_snapshot()
    hits = stats['cache']['hits']
    misses = stats['cache']['misses']
    hr = hits / max(1, hits + misses)
    evictions = stats['cache']['evictions']
    print(f'{name:<25} {hr:>9.1%} {evictions:>10}')

## 7. Measure Dispatch Overhead on GPU

Measure how long the policy dispatch takes relative to actual model inference time.

In [ ]:
# Profile dispatch overhead using the real trace
import time

sched = MoESched()
ir = (
    sched.build('lru_profile')
    .cache(capacity=4, eviction='lru')
    .prefetch(strategy='none')
    .schedule(mode='gpu_only')
    .done()
)
policy = compile_policy(ir)
hook = build_hook(policy)

# Warm up
for entry in trace_data[:100]:
    hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'])

# Time 1000 dispatch calls
N = min(len(trace_data), 10000)
hook = build_hook(compile_policy(ir))  # fresh
t0 = time.perf_counter()
for entry in trace_data[:N]:
    hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'])
elapsed = time.perf_counter() - t0

us_per_call = (elapsed / N) * 1e6
print(f'Dispatch overhead (Python): {us_per_call:.1f} µs/layer ({N} calls)')
print(f'Typical Mixtral layer inference: ~500-2000 µs')
print(f'Estimated overhead: {us_per_call / 1000 * 100:.1f}% of 1ms layer time')

## Next Steps

- [ ] Run with full ShareGPT dataset (100+ prompts)
- [ ] Record traces for DeepSeek-V2-Lite
- [ ] Download traces to local machine for benchmark analysis
- [ ] Proceed to notebook 02 (dispatch profiling with Cython)